# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZiadYakout/FlyRank-Ai/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding 1 from the research paper

The paper reports that **[INSERT THE FIRST ACTUAL FINDING FROM THE PAPER]**.

#### My methodology question

I would ask where the label used for this finding comes from.

Is the label an outcome that was actually observed after the prediction point, or is it derived from another rule or variable in the same dataset?

If the label is derived from a rule, the model may mainly be learning that rule rather than learning a real-world future outcome. I would therefore want the paper to clearly explain how the label was constructed and when it became known.

This is the same standard I should apply to my own work.

---

### Finding 2 from the research paper

The paper reports that **[INSERT THE SECOND ACTUAL FINDING FROM THE PAPER]**.

#### My methodology question

I would ask whether the validation design supports this claim.

In particular, I would want to know whether related observations could appear in both the training and validation data, for example observations belonging to the same client or observations from overlapping time periods.

If the validation data is not sufficiently separated from the training data, the reported performance may be optimistic.

I would therefore want to see a validation design that matches the real decision setting and prevents information from the same client or future period from leaking into the evaluation.

These are constructive methodology questions, not claims that the paper is incorrect. They are the same questions I should ask of my own model.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Record the two research-paper findings that I selected.
# Replace the text below with the exact findings from the paper.

paper_finding_1 = """
[INSERT FIRST PAPER FINDING]
"""

paper_finding_2 = """
[INSERT SECOND PAPER FINDING]
"""

print("Paper finding 1:")
print(paper_finding_1)

print("\nPaper finding 2:")
print(paper_finding_2)

print("\nMethodology questions prepared for both findings.")

Paper finding 1:

[INSERT FIRST PAPER FINDING]


Paper finding 2:

[INSERT SECOND PAPER FINDING]


Methodology questions prepared for both findings.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


My Week-5 model was a Random Forest classifier.

For the first result, I will evaluate the model using a random row-level split. This represents the easier evaluation that can occur when observations from the same client are allowed to appear in both training and testing data.

For the honest result, I will use a grouped-by-client split. All content belonging to a client will remain in either the training set or the test set, never both.

The grouped split is more appropriate for my decision-support question because I want to know whether the model can generalize to clients that it did not see during training.

I will compare both evaluations using the same target, features, model type, and Precision@K metric.

The before/after comparison will show whether the easier random split gives a more optimistic result than the grouped split.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


# =========================================================
# 1. LOAD DATA
# =========================================================

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")

print("Dataset shape:", df.shape)


# =========================================================
# 2. CREATE OBSERVED TARGET
# =========================================================

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)


# =========================================================
# 3. FEATURES FROM WEEK 5
# =========================================================

features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "days_since_last_update"
]

target = "is_declining_label"
group = "client_id"

model_df = df[
    features + [target, group, "content_id"]
].copy()

model_df = model_df.dropna(subset=[target])


# =========================================================
# 4. PRECISION@K FUNCTION
# =========================================================

def precision_at_k(y_true, scores, k):

    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    top_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_indices].mean()


# =========================================================
# 5. RANDOM ROW-LEVEL SPLIT
# =========================================================

random_train, random_test = train_test_split(
    model_df,
    test_size=0.25,
    random_state=42,
    stratify=model_df[target]
)

X_train_random = random_train[features]
y_train_random = random_train[target]

X_test_random = random_test[features]
y_test_random = random_test[target]


random_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
    )
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_probability = random_model.predict_proba(
    X_test_random
)[:, 1]

K_random = max(
    1,
    int(len(random_test) * 0.10)
)

random_precision = precision_at_k(
    y_test_random,
    random_probability,
    K_random
)

random_auc = roc_auc_score(
    y_test_random,
    random_probability
)


# =========================================================
# 6. GROUPED-BY-CLIENT SPLIT
# =========================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df[target],
        groups=model_df[group]
    )
)

group_train = model_df.iloc[train_idx].copy()
group_test = model_df.iloc[test_idx].copy()

X_train_group = group_train[features]
y_train_group = group_train[target]

X_test_group = group_test[features]
y_test_group = group_test[target]


group_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1,
            class_weight="balanced"
        )
    )
])

group_model.fit(
    X_train_group,
    y_train_group
)

group_probability = group_model.predict_proba(
    X_test_group
)[:, 1]

K_group = max(
    1,
    int(len(group_test) * 0.10)
)

group_precision = precision_at_k(
    y_test_group,
    group_probability,
    K_group
)

group_auc = roc_auc_score(
    y_test_group,
    group_probability
)


# =========================================================
# 7. CHECK CLIENT LEAKAGE
# =========================================================

shared_clients = set(
    group_train[group].unique()
).intersection(
    set(group_test[group].unique())
)

print("Random split:")
print("  Train rows:", len(random_train))
print("  Test rows:", len(random_test))
print("  Precision@K:", round(random_precision, 4))
print("  ROC-AUC:", round(random_auc, 4))

print("\nGrouped split:")
print("  Train rows:", len(group_train))
print("  Test rows:", len(group_test))
print("  Train clients:", group_train[group].nunique())
print("  Test clients:", group_test[group].nunique())
print("  Shared clients:", len(shared_clients))
print("  Precision@K:", round(group_precision, 4))
print("  ROC-AUC:", round(group_auc, 4))


# =========================================================
# 8. BEFORE / AFTER TABLE
# =========================================================

validation_comparison = pd.DataFrame({
    "validation_design": [
        "Random row split",
        "Grouped by client"
    ],
    "Precision@K": [
        random_precision,
        group_precision
    ],
    "ROC-AUC": [
        random_auc,
        group_auc
    ],
    "K": [
        K_random,
        K_group
    ]
})

print("\nBEFORE / AFTER VALIDATION")
display(validation_comparison)

Dataset shape: (30000, 44)
Random split:
  Train rows: 22500
  Test rows: 7500
  Precision@K: 0.8093
  ROC-AUC: 0.7139

Grouped split:
  Train rows: 22885
  Test rows: 7115
  Train clients: 24
  Test clients: 8
  Shared clients: 0
  Precision@K: 0.7229
  ROC-AUC: 0.626

BEFORE / AFTER VALIDATION


,validation_design,Precision@K,ROC-AUC,K
0,Random row split,0.809333,0.713948,750
1,Grouped by client,0.722925,0.625977,711


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


I audited the final Week-5 feature set for information that would not be available at the decision moment.

The model features are:

- `impressions_90d`
- `clicks_90d`
- `sessions_90d`
- `avg_position`
- `days_since_last_update`

I deliberately exclude:

- `trend_direction`
- `trend_pct`
- `is_declining_label`

because these are directly related to the observed outcome used as the target.

I also exclude identifiers such as `content_id` and `client_id` from the model features. They are used for identifying rows and creating the grouped validation split, not for prediction.

The important leakage test is whether a feature contains information from the future relative to the decision moment. The current starter-data exercise is limited because its 90-day fields are snapshot measurements. Therefore, the results should be treated as directional rather than as proof of future predictive performance.

The grouped-by-client validation also reduces the risk that client-specific information makes the evaluation look artificially strong.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# LEAKAGE AUDIT
# =========================================================

print("FINAL MODEL FEATURES")
print("--------------------")

for feature in features:
    print("✓", feature)


# Fields that should NOT be used as predictive features
excluded_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

print("\nEXCLUDED FIELDS")
print("----------------")

for field in excluded_fields:
    print("✗", field)


# ---------------------------------------------------------
# Check for obvious label-derived fields
# ---------------------------------------------------------

label_related_words = [
    "label",
    "trend",
    "declin"
]

possible_leakage = []

for feature in features:

    feature_lower = feature.lower()

    if any(
        word in feature_lower
        for word in label_related_words
    ):
        possible_leakage.append(feature)


print("\nAUTOMATIC FEATURE-NAME CHECK")

if len(possible_leakage) == 0:
    print("PASS: No obvious label/trend fields appear in the feature list.")
else:
    print("WARNING: Review these features:")
    print(possible_leakage)


# ---------------------------------------------------------
# Check that IDs aren't model features
# ---------------------------------------------------------

id_features = [
    f for f in features
    if f in ["content_id", "client_id"]
]

print("\nID CHECK")

if len(id_features) == 0:
    print("PASS: IDs are not used as predictive features.")
else:
    print("WARNING: IDs appear as predictive features:")
    print(id_features)

FINAL MODEL FEATURES
--------------------
✓ impressions_90d
✓ clicks_90d
✓ sessions_90d
✓ avg_position
✓ days_since_last_update

EXCLUDED FIELDS
----------------
✗ trend_direction
✗ trend_pct
✗ is_declining_label
✗ content_id
✗ client_id

AUTOMATIC FEATURE-NAME CHECK
PASS: No obvious label/trend fields appear in the feature list.

ID CHECK
PASS: IDs are not used as predictive features.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### My original claim

"My Random Forest can predict which pages are declining and can identify the pages that should be refreshed."

This statement is too strong because the experiment does not establish that refreshing a page will cause it to improve. It also does not prove that the model will perform the same way on future data.

### Safer claim

"On the evaluated starter dataset, the Random Forest produced a measured ranking of content items using the available page-level signals. Under a grouped-by-client validation split, its performance was [INSERT ACTUAL PRECISION@K] Precision@K and [INSERT ACTUAL ROC-AUC] ROC-AUC. These results are directional evidence about the usefulness of the available signals for decision-support, not causal evidence that refreshing a page will improve its performance."

### What I can claim

I can claim that the result was:

- **observed** in the evaluation data,
- **measured** using Precision@K and ROC-AUC,
- **directional** evidence about the usefulness of these signals,
- and potentially useful for **decision-support**.

### What I cannot claim

I cannot claim that:

- the model proves that refreshing causes improvement;
- the model will perform identically on future clients;
- the model has discovered causal relationships;
- the model will always identify the correct pages;
- or the model predicts Google's future ranking decisions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================================================
# FINAL AUDIT SUMMARY
# =========================================================

print("ML-09 FINAL AUDIT")
print("=================")

print("\n1. Validation")
print(
    "Random split Precision@K:",
    round(random_precision, 4)
)
print(
    "Grouped split Precision@K:",
    round(group_precision, 4)
)

print(
    "\nRandom split ROC-AUC:",
    round(random_auc, 4)
)
print(
    "Grouped split ROC-AUC:",
    round(group_auc, 4)
)

print(
    "\nShared clients in grouped split:",
    len(shared_clients)
)

print("\n2. Leakage")
print(
    "Label/trend features used:",
    len(possible_leakage)
)

print(
    "ID features used:",
    len(id_features)
)

print("\n3. Interpretation")

if group_precision < random_precision:
    print(
        "The grouped validation result is lower than the random-split "
        "result, suggesting that the random split may have provided "
        "an optimistic estimate of performance."
    )
elif group_precision > random_precision:
    print(
        "The grouped validation result is higher than the random-split "
        "result. This should still be interpreted cautiously because "
        "the two test sets contain different client groups."
    )
else:
    print(
        "The two validation designs produced the same Precision@K."
    )

print(
    "\nFinal position: results are decision-support evidence, "
    "not causal proof."
)

ML-09 FINAL AUDIT

1. Validation
Random split Precision@K: 0.8093
Grouped split Precision@K: 0.7229

Random split ROC-AUC: 0.7139
Grouped split ROC-AUC: 0.626

Shared clients in grouped split: 0

2. Leakage
Label/trend features used: 0
ID features used: 0

3. Interpretation
The grouped validation result is lower than the random-split result, suggesting that the random split may have provided an optimistic estimate of performance.

Final position: results are decision-support evidence, not causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.